# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**One row = one what?** One row = one content page (`content_hash_id`), for
one client (`client_hash_id`), on one day (`report_date`) — the grain of
`fact_content_daily_performance`.

**Time window:** March 2026 (`month=2026-03`) — a full mid-panel month, not
the sealed final month (`_sample`, which is June 2026 and reserved as a test
month, per the warning on this card).

**What I'd predict/rank:** whether a page is declining, using observed
impression movement within/around this window as the proxy — not any
FlyRank product flag.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Feature (knowable at decision time):** `gsc_impressions`, `gsc_clicks`,
`gsc_avg_position`, `content_created_at` (→ age), `ga4_data_available`.

**Label/proxy:** a future-window decline flag, e.g. April impressions falling
below March impressions for the same page — computed for verification only,
never leaked back in as a feature.

**Context (not a feature or label, just useful for joins/checks):**
`client_hash_id`, `content_hash_id` — pseudonymized IDs used only for
grouping and leakage checks, not treated as meaningful values.

**Excluded, and why:** FlyRank's own product decision flags (`health_score`,
`priority_score`, `action_type`, `refresh_tier`) — these are deliberately not
in the released data, and even if reconstructed, must never be used as a
feature, per the lane guide's observable-only rule. Also excluded: raw
query/URL/title text — only pseudonymized IDs are safe here.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [1]:
import duckdb, os

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{os.environ['HF_TOKEN']}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
fact = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
content = f"read_parquet('{REL}/dim_content.parquet')"

In [3]:
con.sql(f"""
    SELECT content_hash_id, client_hash_id, report_date, COUNT(*) as n
    FROM {fact}
    GROUP BY 1,2,3
    HAVING COUNT(*) > 1
    LIMIT 5
""").show()
# Zero rows returned = grain confirmed: one row per page/client/day.

┌─────────────────┬────────────────┬─────────────┬───────┐
│ content_hash_id │ client_hash_id │ report_date │   n   │
│     varchar     │    varchar     │    date     │ int64 │
└─────────────────┴────────────────┴─────────────┴───────┘
                          0 rows                        



Query 1 — grain check:

In [4]:
con.sql(f"""
    SELECT content_hash_id, client_hash_id, report_date, COUNT(*) as n
    FROM {fact}
    GROUP BY 1,2,3
    HAVING COUNT(*) > 1
    LIMIT 5
""").show()
# Zero rows returned = grain confirmed: one row per page/client/day.

┌─────────────────┬────────────────┬─────────────┬───────┐
│ content_hash_id │ client_hash_id │ report_date │   n   │
│     varchar     │    varchar     │    date     │ int64 │
└─────────────────┴────────────────┴─────────────┴───────┘
                          0 rows                        



Query 2 — row count + date span:

In [5]:
con.sql(f"""
    SELECT COUNT(*) as row_count, MIN(report_date) as min_date, MAX(report_date) as max_date
    FROM {fact}
""").show()

┌───────────┬────────────┬────────────┐
│ row_count │  min_date  │  max_date  │
│   int64   │    date    │    date    │
├───────────┼────────────┼────────────┤
│   9841378 │ 2026-03-01 │ 2026-03-31 │
└───────────┴────────────┴────────────┘



Query 3 — availability, using IS TRUE:

In [6]:
con.sql(f"""
    SELECT COUNT(*) as total_rows,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) as ga4_available_rows
    FROM {fact}
""").show()

┌────────────┬────────────────────┐
│ total_rows │ ga4_available_rows │
│   int64    │       int128       │
├────────────┼────────────────────┤
│    9841378 │             413966 │
└────────────┴────────────────────┘



In [8]:
features = con.sql(f"""
    SELECT f.content_hash_id,
           SUM(f.gsc_impressions) as impressions_march,
           AVG(f.gsc_avg_position) as avg_position_march,
           DATEDIFF('day', c.content_created_date, DATE '2026-03-31') as content_age_days,
           SUM(CASE WHEN f.ga4_data_available IS TRUE THEN 1 ELSE 0 END) as days_with_ga4,
           MAX(f.report_date) as last_seen_date
    FROM {fact} f JOIN {content} c ON f.content_hash_id = c.content_hash_id
    GROUP BY 1, c.content_created_date
""").df()
features.head()

,content_hash_id,impressions_march,avg_position_march,content_age_days,days_with_ga4,last_seen_date
0,content_d9a83099868679ac,2000.0,12.042561,193,0.0,2026-03-31
1,content_c7849d51e853d77a,574.0,10.008971,193,0.0,2026-03-31
2,content_cb70f3d0c0773204,1926.0,3.568652,193,0.0,2026-03-31
3,content_4ef2ba84dfb8d463,904.0,6.845208,193,0.0,2026-03-31
4,content_a8d46b008e9eefb4,96.0,20.455952,193,0.0,2026-03-31


1. `impressions_march` — knowable because GSC impressions log as they happen; complete by month-end.
2. `avg_position_march` — same: position is recorded daily, fully observed by March 31.
3. `content_age_days` — knowable because content_created_at is a fixed past date.
4. `days_with_ga4` — just counts already-passed days, not future behavior.
5. `last_seen_date` — most recent day *within* the observed window, not a future date.

In [10]:
fact_april = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')"
leaky = con.sql(f"SELECT content_hash_id, SUM(gsc_impressions) as impressions_april FROM {fact_april} GROUP BY 1").df()

lf = features.merge(leaky, on="content_hash_id", how="left")
lf["is_declining"] = (lf["impressions_april"] < lf["impressions_march"]).astype(int)
lf["leaky_signal"] = lf["impressions_april"]  # the trap: literally the answer

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

X_leaky = lf[["impressions_march","avg_position_march","leaky_signal"]].fillna(0)
y = lf["is_declining"]
print("WITH leak, AUC:", roc_auc_score(y, LogisticRegression().fit(X_leaky,y).predict_proba(X_leaky)[:,1]))

X_honest = lf[["impressions_march","avg_position_march","content_age_days"]].fillna(0)
print("WITHOUT leak, AUC:", roc_auc_score(y, LogisticRegression().fit(X_honest,y).predict_proba(X_honest)[:,1]))

WITH leak, AUC: 0.9999997873309947
WITHOUT leak, AUC: 0.8215976874673506


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

March 2026 is one month of an unbalanced panel — some clients joined recently
and have little history here, so this slice may under-represent long-tenure
clients. It's also a single month, so seasonal effects specific to March
won't necessarily generalize to other months without further checking.

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.